# 03 — Silver transformation

**Purpose:** turn the raw bronze tables into one clean, typed table with real column names, ready for analysis and modeling.

**Reads from:**
- `kitsune_dev.bronze.syn_dos_features_raw` (115 feature columns as text, `_c0` to `_c114`)
- `kitsune_dev.bronze.syn_dos_labels_raw` (label per packet)

**Where the logic lives:** the transformation functions are in `src/kitsune/` (Python modules), not in this notebook.
This notebook only **orchestrates**: it imports those functions, runs them in order and writes the result.

| File | What it does |
|---|---|
| `src/kitsune/feature_names.py` | Generates the 115 real column names (e.g. `_c36` → `hh_l5_pcc`) |
| `src/kitsune/silver.py` | Casts types, joins features with labels, runs quality checks *(coming next)* |

**Status:** work in progress. Right now this notebook only tests that the modules can be imported.

## Setup: reload modules automatically

**What:** turn on *autoreload*.

**Why:** Python reads a `.py` file only the **first time** you import it, and then keeps that version in memory. If you edit `feature_names.py` afterwards, the notebook would keep using the old version until you restart Python.
With autoreload on, Python checks for changes before running each cell and reloads any module that changed.

In [0]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Test: generate the 115 column names

**What:** import `feature_names()` from `src/kitsune/feature_names.py` and check its output.

**How the names are built:** `{block}_l{window}_{statistic}`, all lowercase.
- `block`: mi, h, hh, hh_jit, hphp (who is talking to whom)
- `window`: the λ value; dots become `_` because a dot in a Spark column name means "field inside a struct" (`0.1` → `0_1`)
- `statistic`: weight, mean, variance, radius, magnitude, cov, pcc

The order follows layout A, verified in `Notebooks/exploration/feature_layout_check`.

**Expected result:** 115 names. Columns 0, 36, 51 and 78 should be `mi_l5_weight`, `hh_l5_pcc`, `hh_l0_1_weight` and `hh_jit_l0_01_mean`.

In [0]:
from src.kitsune.feature_names import feature_names

names = feature_names()
print(len(names))
print(names[0], names[36], names[51], names[78])

115
mi_l5_weight hh_l5_pcc hh_l0_1_weight hh_jit_l0_01_mean
